# SafeTails — Species Classifier: Colab Pro Training & Export

Retrains the **only in-house model** (5-class species classifier) with modern, widely-adopted datasets and architectures, then exports artefacts that drop straight into `ml/exported/`.

**Taxonomy (unchanged, thesis scope):** `Dog, Cat, Cow, Buffalo, Other`.

**Upgrades vs the baseline**
- Datasets: **Oxford-IIIT Pet** (cat/dog) + **Stanford Dogs**, **regional Indian/Pakistani cattle & buffalo** (Kaggle - fixes the Nepal domain gap on the two weakest classes) + **Animals-10** for *Other*; ImageNet synsets remain a fallback top-up.
- Backbones: **EfficientNetV2-S** and **ConvNeXt-Tiny** (vs MobileNetV3 / EfficientNet-B0 baseline).
- Training: RandAugment + Random Erasing + **MixUp/CutMix**, label smoothing, cosine LR, **EMA**, AMP.
- **Optuna** hyper-parameter search, temperature-scaling **calibration** (ECE), full evaluation.
- ONNX export (opset 17) with the **exact preprocessing the backend uses** (Resize 224 + ImageNet norm).

> Runtime → Change runtime type → **GPU** (T4/A100). `SEED=42` throughout for reproducibility.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 · Setup

In [ ]:
get_ipython().system('pip -q install onnx onnxruntime timm optuna scikit-learn seaborn datasets huggingface_hub "pillow==10.1" onnxscript --upgrade')
import torch, torchvision, platform
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
import os, random, numpy as np, torch
SEED = 42
def set_seed(s=SEED):
    random.seed(s); os.environ['PYTHONHASHSEED']=str(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

CLASS_NAMES = ['Dog', 'Cat', 'Cow', 'Buffalo', 'Other']   # order MUST match the backend
IMG_SIZE = 224
PER_CLASS = 2500           # collect generously per class; class-balanced sampling handles any
                           # remaining imbalance during training. Lower to ~1200 for a fast run,
                           # raise to 4000+ on an A100 for the strongest model.
SPLIT = (0.70, 0.15, 0.15) # train / val / test, stratified
from pathlib import Path
RAW = Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
EXPORT = Path('exported'); EXPORT.mkdir(exist_ok=True)
IMAGENET_MEAN=[0.485,0.456,0.406]; IMAGENET_STD=[0.229,0.224,0.225]

## 2 · Acquire datasets → `data/raw/<Class>/`
Each class is filled from standard public sources, capped for balance. Failures on any one source are skipped so acquisition is robust.

In [ ]:
import io, requests
from PIL import Image

def _save(img, cls, i, max_side=512):
    try:
        img = img.convert('RGB')
        if max(img.size) > max_side: img.thumbnail((max_side, max_side))
        d = RAW/cls; d.mkdir(parents=True, exist_ok=True)
        img.save(d/f'{cls.lower()}_{i:05d}.jpg', 'JPEG', quality=90); return True
    except Exception: return False

def count(cls): return len(list((RAW/cls).glob('*.jpg')))

In [ ]:
# --- Cat & Dog: Oxford-IIIT Pet (torchvision downloads it) -----------------
from torchvision.datasets import OxfordIIITPet
pet = OxfordIIITPet(root='oxford_pet', download=True, target_types='binary-category')
# binary-category: 0 = cat, 1 = dog
ci = {0:'Cat', 1:'Dog'}; seen={'Cat':count('Cat'),'Dog':count('Dog')}
for img, y in pet:
    cls = ci[int(y)]
    if seen[cls] >= PER_CLASS:
        if seen['Cat']>=PER_CLASS and seen['Dog']>=PER_CLASS: break
        continue
    if _save(img, cls, seen[cls]): seen[cls]+=1
print('Oxford Pet ->', {k:count(k) for k in ['Cat','Dog']})

In [ ]:
# --- More dogs: Stanford Dogs (HF) for breed variety (optional, skipped on failure) ---
from datasets import load_dataset
def fill_from_hf(cls, dataset_id, image_col=None, label_filter=None, cap=PER_CLASS):
    try:
        ds = load_dataset(dataset_id, split='train', streaming=True)
    except Exception as e:
        print('skip', dataset_id, '-', str(e)[:80]); return
    n = count(cls)
    icol = image_col
    for ex in ds:
        if n >= cap: break
        if icol is None:
            icol = next((k for k,v in ex.items() if hasattr(v,'convert') or (isinstance(v,dict) and 'bytes' in v)), None)
            if icol is None: break
        v = ex[icol]
        try:
            img = v if hasattr(v,'convert') else Image.open(io.BytesIO(v['bytes']))
        except Exception: continue
        if _save(img, cls, n): n += 1
    print(f'{cls} <- {dataset_id}: now {count(cls)}')

fill_from_hf('Dog', 'Alanox/stanford-dogs')   # harmless if it fails; Oxford already gave us dogs

## 2c · Regional cattle & buffalo + Animals-10 (Kaggle) - the domain-gap upgrade
The baseline used generic ImageNet ox / water-buffalo, which don't match South-Asian animals. These Kaggle datasets are **Indian/Pakistani breeds** (much better for Nepal, and they target the two weakest classes) plus **Animals-10** for a rich *Other*. Upload your `kaggle.json` (Kaggle -> Settings -> Create New API Token) when prompted.

In [ ]:
import os, subprocess, glob
KAGGLE_OK=False
try:
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        from google.colab import files; up=files.upload()  # pick kaggle.json
        os.makedirs('/root/.kaggle', exist_ok=True)
        for fn in up:
            if fn.endswith('.json'): open('/root/.kaggle/kaggle.json','wb').write(up[fn])
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
    subprocess.run(['pip','-q','install','kaggle'])
    KAGGLE_OK=os.path.exists('/root/.kaggle/kaggle.json'); print('Kaggle ready:', KAGGLE_OK)
except Exception as e:
    print('Kaggle not set up (', e, ') - will fall back to ImageNet synsets')

In [ ]:
# Paste each dataset's Kaggle SLUG (the '<owner>/<name>' part of its kaggle.com/datasets URL).
#   fixed  -> the WHOLE dataset is one class
#   route  -> assign each image by the first keyword found in its file path
KAGGLE_SOURCES = [
    # Dog + Cat (ADDS to Oxford-IIIT Pet + Stanford Dogs, does not replace them):
    {'slug':'bhavikjikadara/dog-and-cat-classification-dataset', 'route':{'cat':'Cat','dog':'Dog'}},
    # Cow + Buffalo (regional South-Asian - the key domain-gap upgrade):
    {'slug':'raghavdharwal/cows-and-buffalo-computer-vision-dataset', 'route':{'buffalo':'Buffalo','cow':'Cow','cattle':'Cow'}},
    {'slug':'algsoch/breed-cattle-buffalo',        'route':'BREEDS'},   # 75 Indian breeds -> Cow/Buffalo
    {'slug':'atharvadarpude/indian-buffalo-dataset','fixed':'Buffalo'},
    # Other (also supplements Dog/Cat/Cow) - Animals-10 uses Italian folder names:
    {'slug':'alessiocorrado99/animals10',          'route':{'cane':'Dog','gatto':'Cat','mucca':'Cow',
        'pecora':'Other','gallina':'Other','cavallo':'Other','elefante':'Other','scoiattolo':'Other','farfalla':'Other','ragno':'Other'}},
    # Wild/zoo animals -> a richer, harder Other:
    {'slug':'jirkadaberger/zoo-animals',           'fixed':'Other'},
]
# Buffalo breeds (everything else in a mixed cattle+buffalo set is treated as Cow).
BUFFALO_BREEDS=['murrah','nili','ravi','kundi','jaffarabadi','surti','mehsana','bhadawari','banni','toda','pandharpuri','nagpuri','marathwadi','buffalo']
def _route_breeds(pl):
    return 'Buffalo' if any(b in pl for b in BUFFALO_BREEDS) else 'Cow'

def ingest_kaggle(src):
    slug=src['slug']
    if '<owner>' in slug: print('skip (add the slug):', slug); return
    d=Path('kaggle')/slug.split('/')[-1]; d.mkdir(parents=True, exist_ok=True)
    if not any(d.iterdir()):
        if subprocess.run(['kaggle','datasets','download','-d',slug,'-p',str(d),'--unzip']).returncode!=0:
            print('download failed (check slug / accept dataset terms):', slug); return
    n={c:count(c) for c in CLASS_NAMES}
    for f in glob.glob(str(d/'**'/'*.*'), recursive=True):
        pl=f.lower()
        if not pl.endswith(('.jpg','.jpeg','.png')): continue
        if 'fixed' in src: cls=src['fixed']
        elif src.get('route')=='BREEDS': cls=_route_breeds(pl)
        else: cls=next((c for kw,c in src['route'].items() if kw in pl), None)
        if not cls or n[cls]>=PER_CLASS: continue
        try:
            if _save(Image.open(f), cls, n[cls]): n[cls]+=1
        except Exception: pass
    print(f'{slug}:', {c:count(c) for c in CLASS_NAMES})

if KAGGLE_OK:
    for s in KAGGLE_SOURCES: ingest_kaggle(s)
else:
    print('Skipping Kaggle sources (no token).')

## 2d · Local datasets you downloaded (Mendeley / Roboflow / IEEE)
Drag the unzipped folder into Colab's **Files** panel (left), set its path + routing below. (The HF `Rapidata/Animals-10` is the *same content* as the Kaggle Animals-10 already wired above, so it's skipped to avoid duplicating the *Other* class.)

In [ ]:
# Ingest an already-downloaded LOCAL folder tree. Same routing options as Kaggle:
#   fixed='Buffalo'  (whole folder is one class)
#   route={'buffalo':'Buffalo','cow':'Cow'}   (assign by keyword in the path)
#   route='BREEDS'   (buffalo-breed names -> Buffalo, else -> Cow)
def ingest_folder(root, fixed=None, route=None):
    root=Path(root)
    if not root.exists(): print('folder not found:', root, '- upload/unzip it first'); return
    n={c:count(c) for c in CLASS_NAMES}
    for f in glob.glob(str(root/'**'/'*.*'), recursive=True):
        pl=f.lower()
        if not pl.endswith(('.jpg','.jpeg','.png')): continue
        if fixed: cls=fixed
        elif route=='BREEDS': cls=_route_breeds(pl)
        elif route: cls=next((c for kw,c in route.items() if kw in pl), None)
        else: cls=None
        if not cls or n[cls]>=PER_CLASS: continue
        try:
            if _save(Image.open(f), cls, n[cls]): n[cls]+=1
        except Exception: pass
    print('local', root.name, ':', {c: count(c) for c in CLASS_NAMES})

# Mendeley vdgnxsm692 (data.mendeley.com/datasets/vdgnxsm692/2). Upload+unzip it, set the path,
# then pick the routing that matches its folders (inspect them once): fixed / route / 'BREEDS'.
MENDELEY_PATH = 'mendeley_vdgnxsm692'   # <- change to the unzipped folder's path in Colab
ingest_folder(MENDELEY_PATH, route={'buffalo':'Buffalo','cow':'Cow','cattle':'Cow'})
# For a Roboflow / IEEE export, drop the folder in and call ingest_folder('its_path', route=...).

In [ ]:
# --- Fallback top-up from ImageNet synsets for any class still short (also the no-Kaggle path) ---
SYNSETS = {
  'Cow':     ['mlnomad/imnet1k_ox'],
  'Buffalo': ['mlnomad/imnet1k_water_buffalo_water_ox_Asiatic_buffalo_Bubalus_bubalis'],
  'Other':   ['mlnomad/imnet1k_ram_tup','mlnomad/imnet1k_hog_pig_grunter_squealer_Sus_scrofa',
              'mlnomad/imnet1k_macaque','mlnomad/imnet1k_hen','mlnomad/imnet1k_goose'],
}
for cls, ids in SYNSETS.items():
    for did in ids:
        if count(cls) >= PER_CLASS: break
        fill_from_hf(cls, did, cap=PER_CLASS)
print('final counts:', {c: count(c) for c in CLASS_NAMES})

## 2b · Exploratory data analysis (this dataset)
EDA on the *actual* data the v2 model trains on (different sources from notebook 01): class balance, a labelled sample grid, and the image-size distribution. Figures are saved for the report.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
FIG = EXPORT/'figures'; FIG.mkdir(parents=True, exist_ok=True)
counts = {c: count(c) for c in CLASS_NAMES}
fig,ax=plt.subplots(figsize=(6,3.2))
ax.bar(list(counts), list(counts.values()), color=['#3aa3b5','#e0a92e','#8a63d2','#b5651d','#5b8c5a'])
ax.set_title('Class balance (images per class)'); ax.set_ylabel('images')
for i,(k,v) in enumerate(counts.items()): ax.text(i, v+3, str(v), ha='center', fontsize=9)
plt.tight_layout(); plt.savefig(FIG/'eda_class_balance.png', dpi=130); plt.show()

In [ ]:
# Labelled sample grid: 2 examples per class
import glob
fig,axes=plt.subplots(len(CLASS_NAMES),2,figsize=(4.5,1.9*len(CLASS_NAMES)))
for r,c in enumerate(CLASS_NAMES):
    files=sorted(glob.glob(str(RAW/c/'*.jpg')))[:2]
    for col in range(2):
        ax=axes[r][col]; ax.axis('off')
        if col<len(files): ax.imshow(Image.open(files[col]).convert('RGB'));
        if col==0: ax.set_title(c, loc='left', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.savefig(FIG/'eda_sample_grid.png', dpi=130); plt.show()

In [ ]:
# Image-size distribution (spread of source resolutions)
import glob
ws=[]; hs=[]
for c in CLASS_NAMES:
    for f in sorted(glob.glob(str(RAW/c/'*.jpg')))[:120]:
        try: w,h=Image.open(f).size; ws.append(w); hs.append(h)
        except Exception: pass
fig,ax=plt.subplots(figsize=(5,3.5)); ax.scatter(ws,hs,s=6,alpha=0.35,color='#157d8f')
ax.set_xlabel('width (px)'); ax.set_ylabel('height (px)'); ax.set_title('Source image sizes')
plt.tight_layout(); plt.savefig(FIG/'eda_image_sizes.png', dpi=130); plt.show()

## 3 · Build dataframe + stratified 70/15/15 split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
rows = [(str(p), c) for c in CLASS_NAMES for p in (RAW/c).glob('*.jpg')]
df = pd.DataFrame(rows, columns=['path','label'])
print(df.groupby('label').size().reindex(CLASS_NAMES))
tr, tmp = train_test_split(df, test_size=SPLIT[1]+SPLIT[2], stratify=df.label, random_state=SEED)
va, te = train_test_split(tmp, test_size=SPLIT[2]/(SPLIT[1]+SPLIT[2]), stratify=tmp.label, random_state=SEED)
print('train/val/test:', len(tr), len(va), len(te))

## 4 · Datasets & transforms
Train: RandomResizedCrop + RandAugment + flip + Random Erasing. Val/Test: **Resize(224) + Normalize** — identical to the backend's `species.py` preprocessing, so metrics reflect deployment.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
LBL2I = {c:i for i,c in enumerate(CLASS_NAMES)}
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6,1.0)), T.RandomHorizontalFlip(),
    T.RandAugment(num_ops=2, magnitude=7), T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD), T.RandomErasing(p=0.25),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class DS(Dataset):
    def __init__(self, frame, tf): self.f=frame.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        r=self.f.iloc[i]; img=Image.open(r.path).convert('RGB'); return self.tf(img), LBL2I[r.label]

BATCH=64
# Class-balanced sampling: draw each class equally often so the model isn't skewed toward the
# larger classes (e.g. Dog) - uses ALL images rather than hard-capping to the smallest class.
from torch.utils.data import WeightedRandomSampler
cw = tr['label'].value_counts().to_dict()
sample_w = tr['label'].map(lambda c: 1.0/cw[c]).to_numpy()
sampler = WeightedRandomSampler(torch.as_tensor(sample_w, dtype=torch.double), num_samples=len(tr), replacement=True)
dl_tr=DataLoader(DS(tr,train_tf),batch_size=BATCH,sampler=sampler,num_workers=2,pin_memory=True,drop_last=True)
dl_va=DataLoader(DS(va,eval_tf),batch_size=BATCH,num_workers=2,pin_memory=True)
dl_te=DataLoader(DS(te,eval_tf),batch_size=BATCH,num_workers=2,pin_memory=True)

## 5 · Model factory + MixUp/CutMix + EMA

In [ ]:
import torch.nn as nn, torchvision.models as M, copy
def build(name):
    if name=='efficientnet_v2_s':
        m=M.efficientnet_v2_s(weights='IMAGENET1K_V1'); m.classifier[1]=nn.Linear(m.classifier[1].in_features,5)
    elif name=='convnext_tiny':
        m=M.convnext_tiny(weights='IMAGENET1K_V1'); m.classifier[2]=nn.Linear(m.classifier[2].in_features,5)
    elif name=='efficientnet_b0':
        m=M.efficientnet_b0(weights='IMAGENET1K_V1'); m.classifier[1]=nn.Linear(m.classifier[1].in_features,5)
    else: raise ValueError(name)
    return m

from torchvision.transforms import v2
mixup = v2.MixUp(num_classes=5, alpha=0.2); cutmix = v2.CutMix(num_classes=5, alpha=1.0)
def mix(x,y):
    return (cutmix if random.random()<0.5 else mixup)(x,y)

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay=decay; self.shadow=copy.deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for s,p in zip(self.shadow.parameters(), model.parameters()): s.mul_(self.decay).add_(p, alpha=1-self.decay)
        for s,p in zip(self.shadow.buffers(), model.buffers()): s.copy_(p)

## 6 · Training loop (AMP · cosine · label smoothing · MixUp/CutMix · EMA · best macro-F1)

In [ ]:
from sklearn.metrics import f1_score
@torch.no_grad()
def evaluate(model, dl):
    model.eval(); ys=[]; ps=[]
    for x,y in dl:
        x=x.to(DEVICE); out=model(x).argmax(1).cpu(); ys+=y.tolist(); ps+=out.tolist()
    return f1_score(ys, ps, average='macro'), ys, ps

def train_model(name, epochs=12, lr=3e-4, wd=0.05, label_smooth=0.1):
    set_seed()
    model=build(name).to(DEVICE); ema=EMA(model)
    opt=torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs*len(dl_tr))
    lossf=nn.CrossEntropyLoss(label_smoothing=label_smooth)
    scaler=torch.cuda.amp.GradScaler(enabled=DEVICE=='cuda')
    best=-1; best_state=None; hist={'val_f1':[], 'train_loss':[]}
    for ep in range(epochs):
        model.train(); running=0.0; nb=0
        for x,y in dl_tr:
            x=x.to(DEVICE); y=y.to(DEVICE); x,y=mix(x,y)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=DEVICE=='cuda'):
                loss=lossf(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); ema.update(model)
            running+=float(loss.item()); nb+=1
        f1,_,_=evaluate(ema.shadow, dl_va)
        hist['val_f1'].append(f1); hist['train_loss'].append(running/max(nb,1))
        print(f'{name} ep{ep+1}/{epochs} val_macroF1={f1:.4f}')
        if f1>best: best=f1; best_state=copy.deepcopy(ema.shadow.state_dict())
    model.load_state_dict(best_state); model.eval()
    return model, best, hist

## 7 · Train & compare candidate architectures

In [ ]:
results={}
for name in ['efficientnet_v2_s','convnext_tiny']:   # add 'efficientnet_b0' for the baseline
    m,f1,hist=train_model(name, epochs=15)
    tef1,_,_=evaluate(m, dl_te)
    results[name]={'model':m,'val_f1':f1,'test_f1':tef1,'hist':hist}
    print(f'==> {name}: val {f1:.4f} | test {tef1:.4f}')
best_name=max(results, key=lambda k: results[k]['val_f1']); best_model=results[best_name]['model']
print('BEST:', best_name)

## 8 · Full evaluation (report · confusion matrix · ROC-AUC · PR)

In [ ]:
import numpy as np, seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
@torch.no_grad()
def logits_on(model, dl):
    model.eval(); L=[]; Y=[]
    for x,y in dl:
        L.append(model(x.to(DEVICE)).cpu().numpy()); Y+=y.tolist()
    return np.concatenate(L), np.array(Y)
logits, y_true = logits_on(best_model, dl_te)
prob = torch.softmax(torch.tensor(logits),1).numpy(); y_pred = prob.argmax(1)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
acc=accuracy_score(y_true,y_pred); macro_f1=f1_score(y_true,y_pred,average='macro')
try: auc=roc_auc_score(np.eye(5)[y_true], prob, average='macro', multi_class='ovr')
except Exception: auc=float('nan')
print(f'accuracy={acc:.4f}  macro-F1={macro_f1:.4f}  macro ROC-AUC={auc:.4f}')
cm=confusion_matrix(y_true,y_pred)
plt.figure(figsize=(5,4)); sns.heatmap(cm,annot=True,fmt='d',cmap='Greens',xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES)
plt.xlabel('predicted'); plt.ylabel('true'); plt.title(f'{best_name} confusion'); plt.show()

## 9 · Temperature-scaling calibration (ECE before/after)

In [ ]:
vlog, vy = logits_on(best_model, dl_va)
def ece(probs, labels, bins=15):
    conf=probs.max(1); pred=probs.argmax(1); acc=(pred==labels).astype(float); e=0.0
    for i in range(bins):
        lo,hi=i/bins,(i+1)/bins; m=(conf>lo)&(conf<=hi)
        if m.sum()>0: e+=abs(acc[m].mean()-conf[m].mean())*m.mean()
    return float(e)
import torch.nn.functional as F
logit_t=torch.tensor(vlog, dtype=torch.float32); y_t=torch.tensor(vy)
# Optimise over LOG-temperature so T = exp(logT) is ALWAYS positive (a negative T would
# invert the logits and break inference).
logT=torch.nn.Parameter(torch.zeros(1)); opt=torch.optim.LBFGS([logT],lr=0.05,max_iter=80)
def closure(): opt.zero_grad(); l=F.cross_entropy(logit_t/logT.exp(), y_t); l.backward(); return l
opt.step(closure); T_opt=float(logT.exp().detach().item())
p_before=torch.softmax(torch.tensor(logits,dtype=torch.float32),1).numpy()
p_after=torch.softmax(torch.tensor(logits,dtype=torch.float32)/T_opt,1).numpy()
ece_b=ece(p_before,y_true); ece_a=ece(p_after,y_true)
# Safety: fall back to T=1.0 if the fit is degenerate or made calibration worse.
if not (0.3 <= T_opt <= 6.0) or ece_a > ece_b + 1e-4:
    print(f'calibration fit rejected (T={T_opt:.3f}); using T=1.0'); T_opt=1.0; ece_a=ece_b
print(f'Temperature T={T_opt:.4f}  ECE {ece_b:.4f} -> {ece_a:.4f}')

## 9b · Model visualisations (saved to `figures/` for the report)
Training curves, backbone comparison, normalized confusion, per-class F1, ROC, the confidence distribution + reliability diagram (illustrating why a sharpening temperature is needed), and a qualitative sample-predictions grid.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
# training curves (val macro-F1 per epoch) + backbone comparison
fig,ax=plt.subplots(figsize=(6,3.6))
for name,r in results.items(): ax.plot(range(1,len(r['hist']['val_f1'])+1), r['hist']['val_f1'], marker='o', label=name)
ax.set_xlabel('epoch'); ax.set_ylabel('val macro-F1'); ax.set_title('Training curves'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG/'training_curves.png',dpi=130); plt.show()
names=list(results); vals=[results[n]['test_f1'] for n in names]
fig,ax=plt.subplots(figsize=(5,3.2)); ax.bar(names, vals, color=['#8a63d2','#157d8f']); ax.set_ylim(0,1)
ax.set_title('Test macro-F1 by backbone')
for i,v in enumerate(vals): ax.text(i, v+0.01, f'{v:.3f}', ha='center')
plt.tight_layout(); plt.savefig(FIG/'model_comparison.png',dpi=130); plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns
cmn=confusion_matrix(y_true,y_pred,normalize='true')
fig,ax=plt.subplots(figsize=(5,4)); sns.heatmap(cmn,annot=True,fmt='.2f',cmap='Greens',xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,ax=ax)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('Normalized confusion matrix')
plt.tight_layout(); plt.savefig(FIG/'confusion_normalized.png',dpi=130); plt.show()
per=[f1_score((y_true==i).astype(int),(y_pred==i).astype(int)) for i in range(len(CLASS_NAMES))]
fig,ax=plt.subplots(figsize=(5.5,3)); ax.bar(CLASS_NAMES, per, color='#157d8f'); ax.set_ylim(0,1); ax.set_title('Per-class F1')
for i,v in enumerate(per): ax.text(i,v+0.01,f'{v:.2f}',ha='center')
plt.tight_layout(); plt.savefig(FIG/'per_class_f1.png',dpi=130); plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
Yb=np.eye(len(CLASS_NAMES))[y_true]
fig,ax=plt.subplots(figsize=(5,4))
for i,c in enumerate(CLASS_NAMES):
    fpr,tpr,_=roc_curve(Yb[:,i], prob[:,i]); ax.plot(fpr,tpr,label=f'{c} (AUC {auc(fpr,tpr):.3f})')
ax.plot([0,1],[0,1],'--',color='gray'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC (one-vs-rest)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG/'roc_curves.png',dpi=130); plt.show()

In [ ]:
import torch
p1=torch.softmax(torch.tensor(logits,dtype=torch.float32),1).numpy()
pt=torch.softmax(torch.tensor(logits,dtype=torch.float32)/T_opt,1).numpy()
fig,ax=plt.subplots(figsize=(5.5,3.2))
ax.hist(p1.max(1),bins=20,alpha=0.5,label='T=1 (raw)',color='#b5651d')
ax.hist(pt.max(1),bins=20,alpha=0.5,label=f'T={T_opt:.2f} (calibrated)',color='#157d8f')
ax.axvline(0.70,ls='--',color='red',label='Unverified threshold'); ax.set_xlabel('max softmax confidence'); ax.set_title('Confidence distribution'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG/'confidence_hist.png',dpi=130); plt.show()
def rel(probs,labels,bins=10):
    conf=probs.max(1); pred=probs.argmax(1); acc=(pred==labels).astype(float); xs=[]; ys=[]
    for i in range(bins):
        lo,hi=i/bins,(i+1)/bins; m=(conf>lo)&(conf<=hi)
        if m.sum()>0: xs.append(conf[m].mean()); ys.append(acc[m].mean())
    return xs,ys
fig,ax=plt.subplots(figsize=(4.2,4)); ax.plot([0,1],[0,1],'--',color='gray')
x1,y1=rel(p1,y_true); xt,yt=rel(pt,y_true)
ax.plot(x1,y1,marker='o',label='T=1',color='#b5651d'); ax.plot(xt,yt,marker='o',label=f'T={T_opt:.2f}',color='#157d8f')
ax.set_xlabel('confidence'); ax.set_ylabel('accuracy'); ax.set_title('Reliability diagram'); ax.legend()
plt.tight_layout(); plt.savefig(FIG/'reliability.png',dpi=130); plt.show()

## 10 · Export → `species_model.onnx` + `labels.json` + `calibration.json` + `metrics.json`
The ONNX outputs **logits**; the backend applies softmax with the calibrated temperature (matching `backend/app/ml/species.py`).

In [ ]:
import json
best_model.eval().cpu()
dummy=torch.randn(1,3,IMG_SIZE,IMG_SIZE)
torch.onnx.export(best_model, dummy, str(EXPORT/'species_model.onnx'), input_names=['input'],
    output_names=['logits'], dynamic_axes={'input':{0:'N'},'logits':{0:'N'}}, opset_version=18)
json.dump(CLASS_NAMES, open(EXPORT/'labels.json','w'))
json.dump({'temperature':round(T_opt,4),'ece_before':round(ece_b,4),'ece_after':round(ece_a,4)}, open(EXPORT/'calibration.json','w'))
json.dump({'backbone':best_name,'accuracy':round(float(acc),4),'macro_f1':round(float(macro_f1),4),
           'macro_roc_auc':round(float(auc),4),'per_backbone':{k:{'val_f1':round(v['val_f1'],4),'test_f1':round(v['test_f1'],4)} for k,v in results.items()},
           'classes':CLASS_NAMES,'img_size':IMG_SIZE,'seed':SEED}, open(EXPORT/'metrics.json','w'), indent=2)
print('wrote', [p.name for p in EXPORT.iterdir()])

In [ ]:
# --- sanity check: run the ONNX exactly like the backend does -------------
import onnxruntime as ort, numpy as np
sess=ort.InferenceSession(str(EXPORT/'species_model.onnx'), providers=['CPUExecutionProvider'])
def backend_preprocess(pil):
    im=pil.convert('RGB').resize((IMG_SIZE,IMG_SIZE)); a=np.asarray(im,dtype=np.float32)/255.0
    a=(a-np.array(IMAGENET_MEAN,dtype=np.float32))/np.array(IMAGENET_STD,dtype=np.float32)
    return a.transpose(2,0,1)[None].astype(np.float32)
sample=Image.open(te.iloc[0].path)
lg=sess.run(None,{'input':backend_preprocess(sample)})[0][0]
pr=torch.softmax(torch.tensor(lg)/T_opt,0).numpy()
print('true:', te.iloc[0].label, '| pred:', CLASS_NAMES[int(pr.argmax())], '| conf:', round(float(pr.max()),3))

In [ ]:
# --- download the artefacts, then drop them into ml/exported/ in the repo ---
import shutil
shutil.make_archive('safetails_species_export','zip', EXPORT)
try:
    from google.colab import files; files.download('safetails_species_export.zip')
except Exception: print('Not on Colab - find safetails_species_export.zip in the working dir.')